# Chapter 5: Long chains, the Milne length and the coefficient B

*Code cells: MIT licence (`LICENSE`). Text and figures: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang, 2026).*

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'tutorial' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
pd.set_option('display.precision', 8)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
REF = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
LAM, F = REF['lambda'].to_numpy(), REF['F_continuum'].to_numpy()
Z_M, B_COEF, A_GAUSS = 0.71821164, 0.64680, 4/math.sqrt(3*math.pi)
def check(name, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.10g}  (expected {expected:.10g}, tol {tol:g})')
    return ok

## The leading Gaussian law (derived)

$\langle U_\tau U_0\rangle=e^{-\tau}/3$, so the projected position becomes Brownian with diffusivity
$D=\int_0^\infty\langle U_\tau U_0\rangle d\tau=\tfrac13$, i.e. $\langle Z_\lambda^2\rangle\to2\lambda/3$. The mean range of
Brownian motion with variance $\sigma^2\lambda$ over $[0,\lambda]$ is $\sqrt{8\sigma^2\lambda/\pi}$, hence
$$F(\lambda)\sim\sqrt{16\lambda/(3\pi)}=\frac4{\sqrt{3\pi}}\sqrt\lambda .$$
The supplementary material proves this by a martingale functional central limit theorem, with the martingale clock
$Q_\lambda=\int_0^\lambda(1-U_s^2)ds$ of mean $2\lambda/3$. Its variance, computed below, shows that $Q_\lambda/\lambda$
concentrates at $2/3$:

In [2]:
tau, s_, r_ = sp.symbols('tau s r', positive=True)
cov = sp.Rational(4,45)*sp.exp(-3*sp.Abs(s_-r_))          # Cov(U_s^2, U_r^2) from the P_2 harmonic (eigenvalue 3)
VarQ = sp.simplify(2*sp.integrate(sp.integrate(cov.subs(sp.Abs(s_-r_), s_-r_), (r_, 0, s_)), (s_, 0, tau)))
print('Var Q_tau =', sp.factor(VarQ))
print('E Q_tau   = 2 tau/3   (from <1-U^2> = 2/3)')

Var Q_tau = 8*(3*tau*exp(3*tau) - exp(3*tau) + 1)*exp(-3*tau)/405
E Q_tau   = 2 tau/3   (from <1-U^2> = 2/3)


## The kinetic boundary layer and the Milne problem (conditional)

Fig. 3(c) shows that the actual deficit $4\sqrt{\lambda/3\pi}-F$ tends to a constant. Within a fraction of a
persistence length of a new maximum the tangents are not equilibrated: the contour approached the record
moving outward. The bulk diffusive solution therefore matches the kinetic boundary at an effective plane
displaced by an extrapolation length. Near the target the stationary problem is
$$u\,\partial_y\Psi=\mathcal L_u\Psi,\qquad\Psi(0,u>0)=0,\qquad\Psi\sim y-u+z_M\ (y\to\infty),$$
the angular-diffusion Schwarzschild-Milne problem of Amic, Luck and Nieuwenhuizen (1996), with
$z_M=0.71821164$. Both extremes contribute, so $F=4\sqrt{\lambda/3\pi}-2z_M+\dots$

We solve the Milne problem live at increasing angular order with `code/build_boundary_moments.py`,
which eliminates the two stationary modes analytically and matches the decaying modes on the incoming
Gauss nodes.

In [3]:
from build_boundary_moments import stationary_milne, moment_record
rows = []
for m in (16, 32, 64, 128, 256):
    rec = moment_record(stationary_milne(m)); rows.append(rec)
tab = pd.DataFrame(rows)[['m','z_M','M1','M2','M3','B_from_moments','B_lower_bound','B_upper_bound']]
print(tab.to_string(index=False))
print('\nz_M converges as m^-3; the published value is 0.71821164.')
check('z_M at m=256', float(tab.z_M.iloc[-1]), Z_M, 1e-6)

  m        z_M         M1         M2         M3  B_from_moments  B_lower_bound  B_upper_bound
 16 0.71793013 0.66666667 0.47862009 0.37367527      0.64620083     0.44109564     0.70494941
 32 0.71817502 0.66666667 0.47878335 0.37376023      0.64672215     0.44146391     0.70517853
 64 0.71820696 0.66666667 0.47880464 0.37377107      0.64679061     0.44151195     0.70520841
128 0.71821105 0.66666667 0.47880736 0.37377245      0.64679939     0.44151810     0.70521224
256 0.71821156 0.66666667 0.47880771 0.37377262      0.64680051     0.44151888     0.70521272

z_M converges as m^-3; the published value is 0.71821164.
PASS  z_M at m=256: 0.7182115634  (expected 0.71821164, tol 1e-06)


True

## From the boundary layer to the offset, one step at a time

**Step 1: the transform of $F$.** With $H_s=s\widehat q$, the Laplace transform of the kinetic equation is
$(s+u\partial_y-\mathcal L_u)H_s=0$ with $H_s(0,u>0)=1$, and since $F=\int_0^\infty\!\int_{-1}^1q\,du\,dy=2\int_0^\infty\langle q\rangle_u\,dy$,
$$\widehat F(s)=\frac2s\int_0^\infty\langle H_s\rangle_u\,dy .$$

**Step 2: the bulk mode.** Far from the target, $H_s=A\,\psi_\kappa(u)\,e^{-\kappa y}$ with
$(\mathcal L_u+\kappa u)\psi_\kappa=s\psi_\kappa$, $\langle\psi_\kappa\rangle_u=1$. Perturbation theory in $\kappa$ (carried out
symbolically at the end of this chapter) gives $\psi_\kappa=1+\kappa u+\kappa^2(u^2-\tfrac13)/3+\dots$ and
$s=\kappa^2/3-11\kappa^4/135+\dots$, i.e. $\kappa=\sqrt{3s}\,[1+\tfrac{11}{30}s+\dots]$.

**Step 3: the inner expansion.** At fixed $y$, write $H_s=1-\kappa\Psi+\kappa^2\Phi+\dots$ Since $s=\kappa^2/3+\dots$,
the orders $\kappa$ and $\kappa^2$ of the equation are, with $\mathcal T=u\partial_y-\mathcal L_u$,
$$\mathcal T\Psi=0,\quad\Psi(0,u>0)=0;\qquad\qquad\mathcal T\Phi=-\tfrac13,\quad\Phi(0,u>0)=0 .$$

**Step 4: matching.** Expand the bulk mode with amplitude $A=1-z_M\kappa+a_2\kappa^2+\dots$ at fixed $y$:
$$A\,\psi_\kappa e^{-\kappa y}=1-\kappa\,(y-u+z_M)+\kappa^2\big[\tfrac12y^2-yu+z_M(y-u)+a_2+\tfrac13(u^2-\tfrac13)\big]+\dots$$
The order-$\kappa$ term is the far field of $\Psi$: $\Psi\sim y-u+z_M$, which defines the Milne problem and
its extrapolation length $z_M$. The order-$\kappa^2$ term is the far field of $\Phi$, which
contains the constant $a_2$ needed for $B$.

**Step 5: integrate.** The bulk mode contributes $\int_0^\infty A\langle\psi_\kappa\rangle_ue^{-\kappa y}dy=A/\kappa$, and
the boundary-layer part of $\Psi$ contributes $-\kappa I$ with $I=\int_0^\infty\langle\Psi-y+u-z_M\rangle_u\,dy$:
$$\int_0^\infty\langle H_s\rangle_u\,dy=\frac1\kappa-z_M+(a_2-I)\,\kappa+\dots$$

**Step 6: invert.** Inserting $\kappa(s)$,
$\widehat F=\dfrac2{\sqrt3\,s^{3/2}}-\dfrac{2z_M}s+\dfrac{\sqrt\pi B}{\sqrt s}+\dots$ with
$\sqrt\pi B=2\sqrt3\,(a_2-I)-\tfrac{11}{15\sqrt3}$, and the transforms
$s^{-3/2}\leftrightarrow2\sqrt{\lambda/\pi}$, $s^{-1}\leftrightarrow1$, $s^{-1/2}\leftrightarrow1/\sqrt{\pi\lambda}$ give the long-chain expansion:
$F=\dfrac4{\sqrt{3\pi}}\sqrt\lambda-2z_M+\dfrac B{\sqrt\lambda}+\dots$

The cell carries out steps 3-6 symbolically.

In [4]:
u, y, kap, zM, a2, I_, s = sp.symbols('u y kappa z_M a_2 I s')
Lu = lambda f: sp.Rational(1, 2)*sp.diff((1-u**2)*sp.diff(f, u), u)
Top = lambda f: u*sp.diff(f, y) - Lu(f)                                   # the transport operator T
avg = lambda f: sp.integrate(f, (u, -1, 1))/2
# step 4: expansion of the bulk mode at fixed y
psi = 1 + kap*u + kap**2*(u**2 - sp.Rational(1, 3))/3
mode = sp.series((1 - zM*kap + a2*kap**2)*psi*sp.exp(-kap*y), kap, 0, 3).removeO()
Psi_far = -sp.expand(mode.coeff(kap, 1)); Phi_far = sp.expand(mode.coeff(kap, 2))
print('step 4:  far field of Psi =', Psi_far)
print('         far field of Phi =', Phi_far)
# step 3: the far fields solve the first- and second-order equations
print('step 3:  T Psi_far =', sp.simplify(Top(Psi_far)), '     T Phi_far =', sp.simplify(Top(Phi_far)))
# steps 5-6: integrate, insert kappa(s), read off the coefficients
kappa_s = sp.sqrt(3*s)*(1 + sp.Rational(11, 30)*s)
integral = 1/kap - zM + (a2 - I_)*kap
Fhat = sp.expand(sp.series((2/s*integral).subs(kap, kappa_s), s, 0, 0).removeO())
print('step 6:  F_hat(s) =', Fhat)
coefB = sp.simplify(Fhat.coeff(1/sp.sqrt(s)))
print('         coefficient of s^(-1/2) =', coefB, '   minus [2 sqrt(3)(a_2 - I) - 11/(15 sqrt(3))] =',
      sp.simplify(coefB - (2*sp.sqrt(3)*(a2 - I_) - sp.Rational(11, 15)/sp.sqrt(3))))
lam = sp.symbols('lambda', positive=True); sv = sp.symbols('s', positive=True)
for f in (sp.sqrt(lam), sp.Integer(1), 1/sp.sqrt(lam)):
    print(f'         Laplace transform of {str(f):15s} =', sp.laplace_transform(f, lam, sv, noconds=True))

step 4:  far field of Psi = -u + y + z_M
         far field of Phi = a_2 + u**2/3 - u*y - u*z_M + y**2/2 + y*z_M - 1/9
step 3:  T Psi_far = 0      T Phi_far = -1/3


step 6:  F_hat(s) = -2*sqrt(3)*I/sqrt(s) + 2*sqrt(3)*a_2/sqrt(s) - 2*z_M/s - 11*sqrt(3)/(45*sqrt(s)) + 2*sqrt(3)/(3*s**(3/2))
         coefficient of s^(-1/2) = sqrt(3)*(-90*I + 90*a_2 - 11)/45    minus [2 sqrt(3)(a_2 - I) - 11/(15 sqrt(3))] = 0


         Laplace transform of sqrt(lambda)    = sqrt(pi)/(2*s**(3/2))
         Laplace transform of 1               = 1/s
         Laplace transform of 1/sqrt(lambda)  = sqrt(pi)/sqrt(s)


## The moment representation of $B$

Define $h(v)=\Psi(0,-v)$ and $M_n=\int_0^1v^nh\,dv$. Three moment identities of the Milne equation,
$\partial_y\langle u\Psi\rangle=0$, $\partial_y\langle u^2\Psi\rangle=-\langle u\Psi\rangle$,
$\partial_y\langle u^3\Psi\rangle=\langle\Psi\rangle-3\langle u^2\Psi\rangle$, give with the far field
$M_1=\tfrac23$, $M_2=\tfrac23z_M$, and $I=\int_0^\infty\langle\Psi-y+u-z_M\rangle dy=M_3/2-\tfrac15$. The table above
confirms $M_1$ and $M_2$ to machine precision. Second-order matching by reciprocity then gives
(article and supplementary material)
$$\sqrt\pi\,B=\sqrt3\Big(2z_M^2-2M_3+\tfrac{17}{45}\Big),\qquad
B=\sqrt{\tfrac3\pi}\Big(\tfrac23z_M^2+\tfrac{17}{45}-\tfrac43\operatorname{Var}_M(v)\Big),$$
where $p_M(v)=\tfrac32v\,h(v)$ is the normalized boundary-speed distribution with mean $z_M$. Because
$0\le v\le1$, $\operatorname{Var}_M\le z_M(1-z_M)$ and $B$ is bounded on both sides.

In [5]:
rec = rows[-1]
B_formula = math.sqrt(3/math.pi)*(2*rec['z_M']**2 - 2*rec['M3'] + 17/45)
print('B from (2 z^2 - 2 M3 + 17/45)      :', B_formula)
print('B from moment record (variance form):', rec['B_from_moments'])
print('analytic bounds                     :', rec['B_lower_bound'], '<= B <=', rec['B_upper_bound'])
print('small-s transfer fits (article)     : 0.6468 +/- 0.0004')
check('B from moments', rec['B_from_moments'], 0.6468, 1e-4)

B from (2 z^2 - 2 M3 + 17/45)      : 0.6468005065535443
B from moment record (variance form): 0.6468005065535443
analytic bounds                     : 0.4415188776745556 <= B <= 0.7052127198675897
small-s transfer fits (article)     : 0.6468 +/- 0.0004
PASS  B from moments: 0.6468005066  (expected 0.6468, tol 0.0001)


True

## The constant $a_2$ by reciprocity, one step at a time (supplementary material)

**Moment identities.** Multiply $u\,\partial_y\Psi=\mathcal L_u\Psi$ by $1$, $u$ and $u^2$ and average over $u$.
$\mathcal L_u$ is self-adjoint, and $\mathcal L_u1=0$, $\mathcal L_uu=-u$, $\mathcal L_uu^2=1-3u^2$, so
$$\partial_y\langle u\Psi\rangle=0,\qquad\partial_y\langle u^2\Psi\rangle=-\langle u\Psi\rangle,\qquad
\partial_y\langle u^3\Psi\rangle=\langle\Psi\rangle-3\langle u^2\Psi\rangle .$$
At the target $\Psi=0$ for $u>0$, so $\langle u^n\Psi\rangle(0)=\tfrac12\int_{-1}^0u^n\Psi(0,u)\,du=(-1)^nM_n/2$.
In the far field, $\langle u(y-u+z_M)\rangle=-\tfrac13$, $\langle u^2(y-u+z_M)\rangle=(y+z_M)/3$ and
$\langle u^3(y-u+z_M)\rangle=-\tfrac15$. Therefore:

* first identity: $-M_1/2=-\tfrac13$, so $M_1=\tfrac23$;
* second identity: $\langle u^2\Psi\rangle=M_2/2+y/3$ exactly; comparison with the far field gives $M_2=\tfrac23z_M$;
* third identity: its right-hand side is $\langle\Psi\rangle-y-z_M=\langle\Psi-y+u-z_M\rangle$, so integrating from $0$ to
  $\infty$ gives $I=-\tfrac15-(-M_3/2)=M_3/2-\tfrac15$.

**Reciprocity.** $G(y,u)=\Psi(y,-u)$ solves the adjoint problem $-u\partial_yG=\mathcal L_uG$, with $G(0,u<0)=0$ and
$G\sim y+u+z_M$. Using $u\partial_y\Phi=\mathcal L_u\Phi-\tfrac13$ and the self-adjointness of $\mathcal L_u$,
$$\partial_y\langle u\,\Phi\,G\rangle=\langle(\mathcal L_u\Phi-\tfrac13)G\rangle-\langle\Phi\,\mathcal L_uG\rangle=-\tfrac13\langle\Psi\rangle .$$
At $y=0$ the product $\Phi G$ vanishes for every $u$ ($\Phi=0$ for $u>0$, $G=0$ for $u<0$). Integrating to $Y$:
$$\langle u\,\Phi\,G\rangle(Y)=-\tfrac13\int_0^Y\langle\Psi\rangle\,dy=-\tfrac16Y^2-\tfrac13z_MY-\tfrac13I+o(1).$$
The left-hand side can also be evaluated from the far fields of step 4. Equating the constants gives
$a_2=z_M^2-\tfrac4{45}-I$: the second-order problem never has to be solved.

**Assembly.** $a_2-I=z_M^2-\tfrac4{45}-2I=z_M^2-M_3+\tfrac{14}{45}$, so
$\sqrt\pi B=2\sqrt3(a_2-I)-\tfrac{11}{15\sqrt3}=\sqrt3\,(2z_M^2-2M_3+\tfrac{17}{45})$. With the normalized speed
distribution $p_M(v)=\tfrac32vh(v)$ (norm $\tfrac32M_1=1$, mean $\tfrac32M_2=z_M$, second moment $\tfrac32M_3$),
$M_3=\tfrac23[\operatorname{Var}_M(v)+z_M^2]$ and the moment representation of B follows.

The first cell does the algebra symbolically. The second solves the second-order boundary problem
*directly* (which the argument avoids) and compares.

In [6]:
u, y, Y, zM, a2, I_, M3, VarM = sp.symbols('u y Y z_M a_2 I M_3 Var_M')
Lu = lambda f: sp.Rational(1, 2)*sp.diff((1-u**2)*sp.diff(f, u), u)
avg = lambda f: sp.integrate(f, (u, -1, 1))/2
print('L 1 =', Lu(sp.Integer(1)), '   L u =', sp.expand(Lu(u)), '   L u^2 =', sp.expand(Lu(u**2)))
far = y - u + zM
print('far-field moments: <u Psi> =', avg(u*far), '   <u^2 Psi> =', sp.factor(avg(u**2*far)), '   <u^3 Psi> =', avg(u**3*far))
# reciprocity: far-field value of <u Phi G> at y = Y
Phi_far = Y**2/2 - Y*u + zM*(Y - u) + a2 + (u**2 - sp.Rational(1, 3))/3
G_far = Y + u + zM
lhs = sp.expand(avg(u*Phi_far*G_far))
rhs = -Y**2/6 - zM*Y/3 - I_/3
print('<u Phi G> from the far fields =', lhs)
a2_sol = sp.solve(sp.expand(lhs - rhs), a2)[0]
print('a_2 =', sp.expand(a2_sol), '      (Y-dependent terms cancel:', sp.expand(lhs - rhs).has(Y) is False, ')')
# assembly
I_M3 = M3/2 - sp.Rational(1, 5)
sqrtpiB = sp.expand((2*sp.sqrt(3)*(a2 - I_) - sp.Rational(11, 15)/sp.sqrt(3)).subs(a2, a2_sol).subs(I_, I_M3))
print('sqrt(pi) B =', sp.simplify(sqrtpiB/sp.sqrt(3)), 'x sqrt(3)')
B_var = sp.sqrt(3/sp.pi)*(sp.Rational(2, 3)*zM**2 + sp.Rational(17, 45) - sp.Rational(4, 3)*VarM)
print('the moment representation of B minus this, with M_3 = (2/3)(Var_M + z_M^2):',
      sp.simplify(B_var - (sqrtpiB/sp.sqrt(sp.pi)).subs(M3, sp.Rational(2, 3)*(VarM + zM**2))))
lower = sp.simplify(B_var.subs(VarM, zM*(1 - zM))/sp.sqrt(3/sp.pi) - (2*(zM - sp.Rational(1, 3))**2 + sp.Rational(7, 45)))
print('lower bound, Var_M <= z_M(1 - z_M): difference from 2(z_M - 1/3)^2 + 7/45 =', lower)

L 1 = 0    L u = -u    L u^2 = 1 - 3*u**2
far-field moments: <u Psi> = -1/3    <u^2 Psi> = (y + z_M)/3    <u^3 Psi> = -1/5


<u Phi G> from the far fields = -Y**2/6 - Y*z_M/3 + a_2/3 - z_M**2/3 + 4/135
a_2 = -I + z_M**2 - 4/45       (Y-dependent terms cancel: True )
sqrt(pi) B = -2*M_3 + 2*z_M**2 + 17/45 x sqrt(3)
the moment representation of B minus this, with M_3 = (2/3)(Var_M + z_M^2): 0
lower bound, Var_M <= z_M(1 - z_M): difference from 2(z_M - 1/3)^2 + 7/45 = 0


In [7]:
from check_second_order_matching import solve as second_order
chk = pd.DataFrame([second_order(m) for m in (32, 64, 128)])
print(chk[['m', 'z_M', 'M1_minus_2/3', 'M2_minus_2zM/3', 'I_minus_(M3/2-1/5)']].to_string(index=False))
print()
print(chk[['m', 'a2_direct', 'a2_reciprocity', 'a2_difference', 'B_moment_formula', 'B_direct']].to_string(index=False))
check('a_2: direct solution minus reciprocity (largest)', float(chk.a2_difference.abs().max()), 0.0, 1e-10)

  m        z_M    M1_minus_2/3  M2_minus_2zM/3  I_minus_(M3/2-1/5)
 32 0.71817502  1.66533454e-15  1.77635684e-15     -5.75928194e-16
 64 0.71820696 -2.66453526e-15 -1.72084569e-15      1.01307851e-15
128 0.71821105  6.99440506e-15  1.09356968e-14     -2.63677968e-15

  m  a2_direct  a2_reciprocity  a2_difference  B_moment_formula   B_direct
 32 0.44000636      0.44000636 2.22044605e-16        0.64672215 0.64672215
 64 0.44004681      0.44004681 2.22044605e-16        0.64679061 0.64679061
128 0.44005199      0.44005199 4.10782519e-15        0.64679939 0.64679939
PASS  a_2: direct solution minus reciprocity (largest): 4.107825191e-15  (expected 0, tol 1e-10)


True

In the discretized problem the three moment identities and the reciprocity value of $a_2$ hold to
rounding error at every order, because the Gauss quadrature integrates the polynomials involved exactly.
What converges with the order $m$ is $z_M$ and $M_3$, and with them $B$.

**Why $\Psi\ge0$ (supplementary material).** $\mathcal T\Psi=0$ says that $\Psi$ is harmonic for the contour
that moves toward the target plane. Evaluated along such a contour, $\Psi$ is a martingale until the contour
reaches the plane (where $\Psi=0$, since it arrives with $u\ge0$) or a large distance $R$ (where
$\Psi\simeq R$). Optional stopping gives $\Psi(y,u)=\lim_{R\to\infty}R\,\Pr_{(y,u)}\{\text{distance }R\text{ is reached before the plane}\}\ge0$,
the analog of the harmonic function $y$ for Brownian motion killed at zero. In particular $h(v)\ge0$, so
$p_M$ is a probability density and the bounds on $B$ hold.

## The bulk dispersion relation

The slow bulk mode $\psi_\kappa e^{-\kappa y}$ with $(\mathcal L_u+\kappa u)\psi_\kappa=s\psi_\kappa$ is an ordinary
perturbation problem; the $\kappa^4$ coefficient $s_4=-11/135$ enters the constant $17/45$ above.

In [8]:
u, kap = sp.symbols('u kappa')
L = lambda f: sp.Rational(1,2)*sp.diff((1-u**2)*sp.diff(f, u), u)
avg = lambda f: sp.integrate(f, (u, -1, 1))/2
phi1 = u
s2 = avg(u*phi1)
# solve L phi2 = s2 - u phi1 with zero mean, polynomial ansatz
a2, b2 = sp.symbols('a2 b2')
phi2 = a2*u**2 + b2
sol = sp.solve([sp.Poly(sp.expand(L(phi2) - (s2 - u*phi1)), u).coeffs()[0], avg(phi2)], [a2, b2])
phi2 = phi2.subs(sol)
a3, b3 = sp.symbols('a3 b3'); phi3 = a3*u**3 + b3*u
eq = sp.Poly(sp.expand(L(phi3) - (s2*phi1 - u*phi2)), u)
phi3 = phi3.subs(sp.solve(eq.coeffs(), [a3, b3]))
s4 = sp.simplify(avg(u*phi3))
print('phi2 =', phi2, '   phi3 =', phi3)
print('s2 =', s2, '   s4 =', s4, '   (article: 1/3 and -11/135)')
print('<u^2 phi2> =', sp.simplify(avg(u**2*phi2)), '  (4/135)')

phi2 = u**2/3 - 1/9    phi3 = u**3/18 - 5*u/18
s2 = 1/3    s4 = -11/135    (article: 1/3 and -11/135)
<u^2 phi2> = 4/135   (4/135)


## The telegraph process: the same structure, exactly

A particle with speed $c$ whose velocity correlation decays as $c^2e^{-at}$ (reversals at rate $a/2$) has
diffusivity $D=c^2/a$, $\ell_{\rm f}=c/a$ (speed times velocity correlation time), and the exactly solvable mean range
$$F_T(t)=\frac{2c}a\big[g_0(x)+at\{g_0(x)+g_1(x)\}-1\big],\quad x=\frac{at}2,\quad g_\nu(x)=e^{-x}I_\nu(x).$$
Its large-$t$ expansion has the Gaussian range minus twice the one-dimensional extrapolation length $\ell_{\rm f}$:

In [9]:
from scipy.special import ive
c, a = 0.5, 0.75
FT = lambda t: (2*c/a)*(ive(0, a*t/2) + a*t*(ive(0, a*t/2) + ive(1, a*t/2)) - 1)
D, ell = c*c/a, c/a
for t in (1e2, 1e3, 1e4, 1e5):
    resid = FT(t) - (4*math.sqrt(D*t/math.pi) - 2*ell)
    print(f't={t:>8g}  F_T - [4 sqrt(Dt/pi) - 2 l] = {resid:.6f}   x sqrt(t) = {resid*math.sqrt(t):.5f}   (l/sqrt(pi a) = {ell/math.sqrt(math.pi*a):.5f})')
# symbolic: large-x expansion of e^{-x} I_nu(x) = (2 pi x)^{-1/2} [1 + (1-4nu^2)/(8x) + ...]
x = sp.symbols('x', positive=True); cs, as_ = sp.symbols('c a', positive=True)
g0 = (2*sp.pi*x)**sp.Rational(-1,2)*(1 + sp.Rational(1,8)/x + sp.Rational(9,128)/x**2)
g1 = (2*sp.pi*x)**sp.Rational(-1,2)*(1 - sp.Rational(3,8)/x - sp.Rational(15,128)/x**2)
expr = (2*cs/as_)*(g0 + 2*x*(g0+g1) - 1)
y = sp.symbols('y', positive=True)            # y = 1/x
ser = sp.series(expr.subs(x, 1/y), y, 0, 2).removeO()
print('\nF_T ~', sp.expand(ser), '   with y = 2/(a t)')
print('   i.e. (2c/a)[ sqrt(8x/pi) - 1 + 1/(2 sqrt(2 pi x)) + ... ] = 4 sqrt(D t/pi) - 2 l + l/sqrt(pi a t) + ...')
print('short-time: F_T = c t - a c t^2/8 + ..., check:', (c*0.01 - FT(0.01))/0.01**2, 'vs', a*c/8)

t=     100  F_T - [4 sqrt(Dt/pi) - 2 l] = 0.043504   x sqrt(t) = 0.43504   (l/sqrt(pi a) = 0.43431)
t=    1000  F_T - [4 sqrt(Dt/pi) - 2 l] = 0.013736   x sqrt(t) = 0.43439   (l/sqrt(pi a) = 0.43431)
t=   10000  F_T - [4 sqrt(Dt/pi) - 2 l] = 0.004343   x sqrt(t) = 0.43432   (l/sqrt(pi a) = 0.43431)
t=  100000  F_T - [4 sqrt(Dt/pi) - 2 l] = 0.001373   x sqrt(t) = 0.43431   (l/sqrt(pi a) = 0.43431)

F_T ~ sqrt(2)*c*y**(3/2)/(32*sqrt(pi)*a) + sqrt(2)*c*sqrt(y)/(2*sqrt(pi)*a) - 2*c/a + 4*sqrt(2)*c/(sqrt(pi)*a*sqrt(y))    with y = 2/(a t)
   i.e. (2c/a)[ sqrt(8x/pi) - 1 + 1/(2 sqrt(2 pi x)) + ... ] = 4 sqrt(D t/pi) - 2 l + l/sqrt(pi a t) + ...
short-time: F_T = c t - a c t^2/8 + ..., check: 0.04681647484309527 vs 0.046875


**Conclusion of the chapter.** The WLC offset $-2z_Mp$ is conditional on the matching assumption; the
telegraph case shows that the structure "Brownian range minus twice the Milne extrapolation length"
holds exactly in a solvable persistent walk, and the spectral spans of Chapter 6 agree with the
three-term expansion to $4\times10^{-8}$ relative for $\lambda\ge5000$. The planar run-and-tumble particle obeys the same
rule exactly (discussed in the article). Neither is a proof of the matching.